## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到 `solutions/` (这样 `from attention.mha import ...` 这种导入能直接生效)。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd into `solutions/`, turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        # already inside a chapter folder — climb out
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the confidence chapter's reference .pt files live
control_folder = 'confidence/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 5 章 · Confidence

推理完拿到坐标后还要给每个原子 / 每个 token 对一个**置信度估计**。AF3 的置信头输出 4 路:

| 量 | 含义 | 维度 |
|---|---|---|
| pLDDT | per-atom local-distance-difference test | per atom |
| PAE | predicted aligned error | per token pair |
| PDE | predicted distance error | per token pair |
| resolved | 可解析 / 未解析二分类 | per atom |

ConfidenceHead 整体由前面章节的零件搭起来 (内部跑一个小型 PairformerStack), 我们本章只单独验证 DistogramHead——置信度系统的入口。

## 5.1 DistogramHead (算法 1 第 17 行)

打开 `confidence/distogram_head.py`，把 `forward` 的 TODO 填好。

DistogramHead 把 pair 表示通过一个零初始化的线性层投到 64 个距离 bin 上, 再对 (i, j) 做对称化。零初始化意味着训练初期输出近似均匀分布。

In [ ]:
from confidence.distogram_head import DistogramHead
from confidence.control_values.confidence_checks import (
    c_z, no_bins, test_inputs,
    test_module_shape, test_module_forward,
)

dh = DistogramHead(c_z=c_z, no_bins=no_bins)
test_module_shape(dh, 'distogram_head', control_folder)
test_module_forward(
    dh, 'distogram_head',
    inputs=(test_inputs['z'],),
    output_names='out',
    control_folder=control_folder,
)
print('DistogramHead ✓')

## 章节小结

ConfidenceHead 本身的 `forward` 与 `memory_efficient_forward` 在 `confidence_head.py` 里有详细 TODO, 端到端 notebook 会把它们整体跑一遍。本章重点只是让你理解DistogramHead 这个最简单的 head 长什么样子。